# CS383: Data Science and Machine Learning
## Lecture 11 Exercises — Cross-Validation, Pipelines, Tuning, and Regularization

Fill in every `__________` blank, then run all cells top to bottom. When you've completed this
notebook, download it (File → Save and Export Notebook As → Notebook (.ipynb), or the **Download**
button in the toolbar) and submit it on BrightSpace under **Lecture 11 Exercise** as a Jupyter Notebook
(.ipynb) file.

### Setup — NYC restaurant inspections

Run this first — same row-level dataset from Lecture 10: `is_critical` and `score` are both useful
targets here, `boro` / `cuisine_top` are the features.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import accuracy_score, classification_report, mean_absolute_error, mean_squared_error, r2_score

try:
    raw_path = os.path.expanduser("~/shared/restaurant_inspections_snapshot.csv")
    inspections_df = pd.read_csv(raw_path)
    inspections_df["score"] = pd.to_numeric(inspections_df["score"], errors="coerce")
    inspections_df = inspections_df.dropna(subset=["score", "boro", "cuisine_description"]).reset_index(drop=True)
    inspections_df["is_critical"] = (inspections_df["critical_flag"] == "Critical").astype(int)

    top_cuisines = inspections_df["cuisine_description"].value_counts().nlargest(10).index
    inspections_df["cuisine_top"] = np.where(
        inspections_df["cuisine_description"].isin(top_cuisines), inspections_df["cuisine_description"], "Other"
    )
    live = True

except Exception:
    rng = np.random.default_rng(383)
    n = 4000
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]
    cuisines_clean = ["American", "Chinese", "Italian", "Mexican", "Pizza", "Japanese"]

    inspections_df = pd.DataFrame({
        "boro": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "cuisine_top": rng.choice(cuisines_clean, size=n),
        "score": rng.integers(0, 71, size=n),
    })
    inspections_df["is_critical"] = rng.integers(0, 2, size=n)
    live = False

print(f"{'Shared snapshot' if live else 'Offline fallback'} data: {len(inspections_df):,} violation records")
inspections_df[["boro", "cuisine_top", "score", "is_critical"]].head()

---

### Step 1 — Split, check the baseline, then tune a k-NN classifier with GridSearchCV

Predict `is_critical` from `score`, `boro`, and `cuisine_top`. Split first (`test_size=0.2`,
`random_state=383`, `stratify=y`), and check the majority-class baseline from `y_train` before doing
anything else. Then build a `Pipeline` with a `ColumnTransformer` (scale `score`, one-hot the two
categoricals -- k-NN cares about feature scale, unlike the trees in Lecture 10) and a
`KNeighborsClassifier`. Use `GridSearchCV` to search `n_neighbors` over `[3, 5, 9, 15, 21]` with
`cv=5, scoring="accuracy"`. Fit it on the training data only.

In [ ]:
X = inspections_df[["score", "boro", "cuisine_top"]]
y = inspections_df["is_critical"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=383, stratify=y)

majority_class = y_train.mode()[0]
majority_baseline = (y_test == majority_class).mean()
print(f"Majority-class baseline: {majority_baseline:.3f}")

knn_preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), ["score"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["boro", "cuisine_top"]),
])

knn_pipeline = Pipeline([
    ("preprocessor", knn_preprocessor),
    ("classifier", KNeighborsClassifier()),
])

param_grid = {"classifier____________": [3, 5, 9, 15, 21]}
knn_grid = GridSearchCV(knn_pipeline, param_grid, cv=5, scoring="accuracy")
knn_grid.fit(X_train, y_train)

print("Best n_neighbors:", knn_grid.best_params_["classifier__n_neighbors"])
print(f"Best CV accuracy: {knn_grid.best_score_:.3f}")

### Step 2 — Evaluate on the test set, against the baseline

This is the one time you touch `X_test` / `y_test` in this exercise -- do it now that tuning is
finished.

In [ ]:
y_pred = knn_grid.__________(X_test)
print(f"Test accuracy: {accuracy_score(y_test, y_pred):.3f}  (baseline: {majority_baseline:.3f})")

### Step 3 — Switch targets: tune a Ridge regression on `score`

Now predict the raw inspection `score` (continuous) from `boro` and `cuisine_top`, using `Ridge`. Split
again (a fresh split for a fresh target), build a `Pipeline` with the same kind of `ColumnTransformer` and
a `Ridge()`, then `GridSearchCV` over `alpha = [0.1, 1, 10, 50, 100]` with `cv=5, scoring="r2"`.

In [ ]:
X_reg = inspections_df[["boro", "cuisine_top"]]
y_reg = inspections_df["score"]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=383)

ridge_preprocessor = ColumnTransformer(transformers=[
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["boro", "cuisine_top"]),
])

ridge_pipeline = Pipeline([
    ("preprocessor", ridge_preprocessor),
    ("regressor", __________()),
])

param_grid_ridge = {"regressor__alpha": [0.1, 1, 10, 50, 100]}
ridge_grid = GridSearchCV(ridge_pipeline, param_grid_ridge, cv=5, scoring="r2")
ridge_grid.fit(X_train_reg, y_train_reg)

print("Best alpha:", ridge_grid.best_params_["regressor__alpha"])
print(f"Best CV R²: {ridge_grid.best_score_:.3f}")

### Step 4 — Evaluate the Ridge model, against a plain baseline

Fit a plain `LinearRegression` (no regularization) through the same `Pipeline` shape as a baseline, then
compare its test R² to the tuned Ridge model's test R².

In [ ]:
baseline_pipeline = Pipeline([
    ("preprocessor", ridge_preprocessor),
    ("regressor", LinearRegression()),
])
baseline_pipeline.fit(X_train_reg, y_train_reg)
baseline_r2 = r2_score(y_test_reg, baseline_pipeline.predict(X_test_reg))

ridge_test_r2 = __________(y_test_reg, ridge_grid.predict(X_test_reg))

print(f"Baseline (plain LinearRegression) test R²: {baseline_r2:.3f}")
print(f"Tuned Ridge test R²:                    {ridge_test_r2:.3f}")

### Step 5 — Explain it back

In 2-3 sentences: did tuning `n_neighbors` and `alpha` with `GridSearchCV` actually beat a reasonable
default guess by much? What would you tell a teammate about *why* the preprocessing had to live inside
the `Pipeline` here, in your own words?

**Your explanation:**

---

## Exercise 2 — Reflection (Exit Ticket)

Answer the following in your own words.

1. Why is a single train/test split's score not enough to trust on its own? What does cross-validation
   give you that one split can't?
2. Why does preprocessing need to live inside a `Pipeline` *before* you cross-validate or run
   `GridSearchCV`, rather than being applied by hand first?
3. In your own words, what's the difference between what Ridge and Lasso do to a model's coefficients?
4. Suppose `GridSearchCV` picks `alpha=0.1` as the best regularization strength, and 0.1 was the
   *smallest* value in your grid. What should you do next, and why?
5. What question do you still have about cross-validation, Pipelines, or regularization heading into the
   midterm?

**Your responses:**

1.
2.
3.
4.
5. 

## Optional Challenge

Apply the same regularized-regression workflow to NYC 311's `resolution_time_hours` -- the same target
from the lecture -- but with **Lasso** instead of Ridge.

### Setup — NYC 311

In [ ]:
import os

try:
    raw_path = os.path.expanduser("~/shared/nyc311_snapshot.csv")
    complaints_df = pd.read_csv(raw_path).head(8000)
    complaints_df["created_date"] = pd.to_datetime(complaints_df["created_date"])
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live = True

except Exception:
    rng = np.random.default_rng(383)
    n = 8000
    complaint_types = ["Noise - Residential", "Illegal Parking", "HEAT/HOT WATER",
                        "Blocked Driveway", "Street Condition", "Water System",
                        "PAINT/PLASTER", "Damaged Tree", "Sewer", "Rodent"]
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]

    n_days = 90
    days = pd.date_range("2026-01-01", periods=n_days, freq="D")
    day_idx = rng.integers(0, n_days, size=n)
    created = days[day_idx] + pd.to_timedelta(rng.integers(0, 24 * 60, size=n), unit="m")
    still_open = rng.random(n) < 0.05
    resolution_hours = rng.gamma(shape=1.5, scale=22.0, size=n)
    closed = created + pd.to_timedelta(resolution_hours, unit="h")

    complaints_df = pd.DataFrame({
        "complaint_type": rng.choice(complaint_types, size=n),
        "borough": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "created_date": created,
        "closed_date": np.where(still_open, pd.NaT, closed),
    })
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live = False

complaints_df["resolution_time_hours"] = (
    complaints_df["closed_date"] - complaints_df["created_date"]
).dt.total_seconds() / 3600
complaints_df["hour_filed"] = complaints_df["created_date"].dt.hour
complaints_df = complaints_df.dropna(subset=["resolution_time_hours"]).reset_index(drop=True)
standard_boroughs = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]
complaints_df = complaints_df[complaints_df["borough"].isin(standard_boroughs)].reset_index(drop=True)
top_types = complaints_df["complaint_type"].value_counts().nlargest(15).index
complaints_df["complaint_grouped"] = complaints_df["complaint_type"].where(
    complaints_df["complaint_type"].isin(top_types), "Other"
)

print(f"{'Shared snapshot' if live else 'Offline fallback'} data: {len(complaints_df):,} 311 records")

### Step 1 — Split, build, and tune

Use `hour_filed`, `borough`, `complaint_grouped` as features and `resolution_time_hours` as the target.
Split (`test_size=0.2`, `random_state=383`). Build a `Pipeline` (`StandardScaler` on `hour_filed`,
`OneHotEncoder` on the two categoricals, then `Lasso()`), and `GridSearchCV` over
`alpha = [0.1, 1, 10, 50, 100, 300]` with `cv=5, scoring="r2"`.

In [ ]:
# Your code here


### Step 2 — Evaluate on the test set

In [ ]:
# Your code here


### Step 3 — Reflect

Did Lasso zero out any coefficients here? (Hint: `lasso_grid.best_estimator_.named_steps["regressor"].coef_`,
matched up against `lasso_grid.best_estimator_.named_steps["preprocessor"].get_feature_names_out()`.) What
would that mean about the corresponding feature category, if anything did get zeroed?

**Your answer:**

### Big idea
> Tuning a model's hyperparameters and adding a regularization penalty are not a substitute for good
> features or enough data -- they are safeguards against overfitting what you already have. The
> discipline behind them (`Pipeline`, cross-validation, holding out a test set touched exactly once) is
> what makes any of these numbers trustworthy in the first place. A well-tuned model evaluated
> dishonestly is worse than a simple model evaluated honestly.